# Week 5 Lab: Silver Layer

In this lab you will build the Silver layer of our bookstore's medallion architecture.

You'll clean, normalize, and conform the raw Bronze data into Third Normal Form (3NF) tables. This means:
- Eliminating redundancy (customer data gets its own table)
- Flattening nested data (JSON arrays become rows)
- Unifying multiple sources (online + instore orders become one table)
- Applying data quality checks (rejecting invalid or empty values)

## Setup: Set Your Catalog

**IMPORTANT:** Before running any queries, set your assigned catalog name.

⚠️ **CI Compatibility Warning**

The CI/CD tests run your SQL against a local Spark environment that is stricter than Databricks. To avoid test failures:

- **Do not include your catalog name in table references** — write `bronze.online_orders`, not `your_student_catalog.bronze.online_orders`

In [0]:
-- TODO: Replace with your assigned catalog name
USE CATALOG pushti_shah;


## Silver Data Model — 3NF

```
┌──────────────────┐     ┌─────────────────────────┐     ┌──────────────────┐
│ silver.customers │◄────│      silver.orders      │────►│  silver.stores   │
├──────────────────┤     ├─────────────────────────┤     ├──────────────────┤
│ PK email         │     │ PK order_id             │     │ PK store_nbr     │
│    name          │     │ PK order_channel        │     │    name          │
│    address       │     │    order_datetime       │     │    address       │
│    city          │     │ FK customer_email       │     │    city          │
│    state         │     │ FK store_nbr            │     │    state         │
│    zip           │     │    payment_method       │     │    zip           │
└──────────────────┘     │    total_amount         │     └──────────────────┘
                         │    cashier_name         │
                         └────────────┬────────────┘
                                      │
                                      │ 1:M
                                      ▼
                         ┌─────────────────────────┐     ┌──────────────────────┐
                         │   silver.order_items    │────►│    silver.books      │
                         ├─────────────────────────┤     ├──────────────────────┤
                         │ PK,FK order_id          │     │ PK isbn              │
                         │ PK,FK order_channel     │     │    title             │
                         │ PK,FK isbn              │     │    author            │
                         │     quantity            │     │ FK category_id       │
                         │     unit_price          │     └─────────┬────────────┘
                         └─────────────────────────┘               │
                                                                   ▼
                                                     ┌──────────────────────┐
                                                     │ silver.categories    │
                                                     ├──────────────────────┤
                                                     │ PK category_id       │
                                                     │    category_name     │
                                                     │ FK parent_category_id│◄─┐
                                                     └──────────────────────┘  │
                                                              │  self-ref      │
                                                              └────────────────┘
```

---
## Prerequisites

Before running this notebook:
1. Run the `create_silver` DDL notebook in this week's folder to create the target Silver tables.
2. Make sure the Bronze layer is populated (Week 4 lab).

---
## Step 1: Build silver.stores

Source: `bronze.stores`

A straightforward load — the column names already match between bronze and silver, so no renaming is needed. We just drop the audit columns (`ingestion_timestamp`, `source_filename`) since they belong to the Bronze layer.

Merge stores from `bronze.stores` into `silver.stores` by using a subquery in the USING clause, matching on `store_nbr`. The subquery selects only the business columns, which drops the audit columns cleanly without any extra steps.

In [0]:
-- @test:silver_stores_merge
-- TODO: Write SQL to implement the silver_stores_merge transformation
MERGE INTO silver.stores AS target
  USING (
      SELECT store_nbr, name , address, city , state , zip	
      FROM bronze.stores
  ) AS source
  ON target.store_nbr= source.store_nbr
  WHEN not MATCHED THEN INSERT * ;

---
## Step 2: Build silver.categories

Source: `bronze.categories`

The categories table is a self-referential hierarchy with three levels: category → genre → subgenre. Each row has a `parent_category_id` pointing to its parent (empty string for top-level categories).

In a normalized (3NF) model like silver, hierarchies are stored as a single self-referential table. To find a book’s top-level category, you must join through the hierarchy:
`books → categories (subgenre) → categories (genre) → categories (category)`

This is the trade-off of normalization — no redundancy, but more complex queries. In the gold layer (Week 6), we’ll flatten this hierarchy into denormalized columns (`subgenre`, `genre`, `category`) on `dim_book`.

Merge categories from `bronze.categories` into `silver.categories` by using a subquery in the USING clause, matching on `category_id`. The subquery selects only the business columns, which drops the audit columns cleanly without any extra steps.

In [0]:
-- @test:silver_categories_merge
-- TODO: Write SQL to implement the silver_categories_merge transformation
MERGE INTO silver.categories AS target
  USING (
      SELECT category_id, category_name , parent_category_id
      FROM bronze.categories
  ) AS source
  ON target.category_id= source.category_id
  WHEN not MATCHED THEN INSERT * ;

---
## Step 3: Build silver.books

Source: `bronze.books`

This is a straightforward load with data quality checks applied. We:
- **Trim** whitespace from all string fields
- **Reject** rows where `isbn` or `title` is null or empty
- **Validate** that `isbn` starts with '978-'
- **Drop** the audit columns (`ingestion_timestamp`, `source_filename`) since they belong to the Bronze layer

Merge books from `bronze.books` into `silver.books` by using a subquery in the USING clause, matching on `isbn`. The subquery selects only the business columns and filters out any rows that don’t meet our criteria, so only valid rows make it through.

In [0]:
-- @test:silver_books_merge
-- TODO: Write SQL to implement the silver_books_merge transformation
MERGE INTO silver.books AS target
  USING (
      SELECT isbn ,
    title ,
    author ,
    category_id	
      FROM bronze.books
  ) AS source
  ON target.isbn= source.isbn
  WHEN not MATCHED THEN INSERT * ;

---
## Step 4: Build silver.customers

Source: `bronze.online_orders`

There is no customers CSV — customer data is embedded on each online order. A single customer may have placed multiple orders, and their name or address may have changed between orders.

A pre-built view `most_recent_online_orders` is provided below — it returns one row per customer using `ROW_NUMBER()`, selecting the fields from their most recent order. Use it as the source for your MERGE.

This view is pre-built — run it before your MERGE. It uses `ROW_NUMBER()` to select the most recent order per customer.

In [0]:
-- @test:silver_customers_most_recent_view
CREATE OR REPLACE TEMPORARY VIEW most_recent_online_orders AS
SELECT
  customer_email AS email,
  customer_name AS name,
  customer_address AS address,
  customer_city AS city,
  customer_state AS state,
  customer_zip AS zip
FROM (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY customer_email ORDER BY order_timestamp DESC) AS rn
  FROM bronze.online_orders
)
WHERE rn = 1. ; 


Merge derived customer data into `silver.customers`, matching on `email`. Use `most_recent_online_orders` directly as the source.

In [0]:
-- @test:silver_customers_merge
-- TODO: Write SQL to implement the silver_customers_merge transformation
-- HINT: most_recent_online_orders is your USING source
MERGE INTO silver.customers AS target
  USING most_recent_online_orders AS source
  ON target.email= source.email
  WHEN not MATCHED THEN INSERT * ;

In [0]:
select * from silver.customers

---
## Step 5: Build silver.orders

Sources: `bronze.online_orders` + `bronze.instore_orders`

The Bronze layer has two separate order tables from two source systems. In Silver we unify them into a single `silver.orders` table with:
- An `order_channel` column (`'online'` or `'in-store'`) to distinguish the source
- A composite primary key of `(order_id, order_channel)` — since order IDs could overlap between systems
- Placeholder values to fill missing data:
  - Online orders have no store, so `store_nbr` = `'online'` (string literal)
  - In-store orders don't always have a customer email — if the email is NULL, `customer_email` is set to `'in-store'` (via CASE)
- Renamed timestamp columns to a common `order_datetime`
- Customer detail fields dropped (they now live in `silver.customers`)

Create a temporary view called `orders_unified` that combines `bronze.online_orders` and `bronze.instore_orders` into a single unified schema using `UNION ALL`.

Use the column mapping below — every column in `silver.orders` must appear in both SELECT lists, in the same order:

| `silver.orders` column | From `bronze.online_orders` | From `bronze.instore_orders` |
|---|---|---|
| `order_id` | `order_id` | `order_id` |
| `order_channel` | `'online'` (string literal) | `'in-store'` (string literal) |
| `order_datetime` | `order_timestamp` | `transaction_timestamp` |
| `customer_email` | `customer_email` | `customer_email` if present, or `'in-store'` if NULL |
| `store_nbr` | `'online'` (string literal) | `store_nbr` |
| `payment_method` | `payment_method` | `payment_method` |
| `total_amount` | `total_amount` | `total_amount` |
| `cashier_name` | `CAST(NULL AS STRING)` (online orders have no cashier) | `cashier_name` |

In [0]:
-- @test:silver_orders_unified_view
-- TODO: Write SQL to implement the silver_orders_unified_view transformation
create or replace temporary view silver_orders_unified_view as
select 
  order_id, 
  'online' as order_channel, 
   order_timestamp as order_datetime, 
   customer_email, 
   'online' as store_nbr, 
   payment_method, 
   total_amount, 
   CAST(NULL AS STRING) as cashier_name 
from bronze.online_orders
union all
select 
  order_id, 
  'in-store' as order_channel, 
   transaction_timestamp as order_datetime, 
   (case when customer_email is null then 'in-store' else customer_email end) as customer_email, 
   store_nbr, 
   payment_method, 
   total_amount, 
   cashier_name 

 from bronze.instore_orders

Merge the unified orders into `silver.orders` using the `orders_unified` temporary view created above, matching on the composite key `(order_id, order_channel)`.

In [0]:
-- @test:silver_orders_merge
-- TODO: Write SQL to implement the silver_orders_merge transformation
MERGE INTO silver.orders AS target
USING silver_orders_unified_view AS source
ON target.order_id = source.order_id
and target.order_channel = source.order_channel
WHEN NOT MATCHED THEN INSERT *


---
## Step 6: Build silver.order_items

Sources: `bronze.online_orders` + `bronze.instore_orders` (the `items` JSON column)

In Bronze, each order has an `items` column containing a JSON array of line items:
```json
[{"isbn": "978-...", "title": "...", "quantity": 1, "unit_price": 49.99}, ...]
```

In Silver we **explode** this array so each line item becomes its own row. This is normalization — instead of a nested structure, we get a flat table that can be joined and aggregated with standard SQL.

We use:
- `FROM_JSON` to parse the JSON string into a typed array of structs
- `EXPLODE` (via `LATERAL VIEW`) to turn each array element into a row
- We drop `title` from the struct since it's redundant with `silver.books`

This view is pre-built — run it before your MERGE. It uses `FROM_JSON` and `LATERAL VIEW EXPLODE` to turn each JSON array element into its own row.

In [0]:
-- @test:silver_order_items_exploded_view
CREATE OR REPLACE TEMPORARY VIEW order_items_exploded AS

SELECT
  order_id,
  'online' AS order_channel,
  item.isbn,
  item.quantity,
  CAST(item.unit_price AS DECIMAL(10, 2)) AS unit_price
FROM bronze.online_orders
  LATERAL VIEW EXPLODE(
    FROM_JSON(items, 'ARRAY<STRUCT<isbn: STRING, title: STRING, quantity: INT, unit_price: DOUBLE>>')
  ) t AS item

UNION ALL

SELECT
  order_id,
  'in-store' AS order_channel,
  item.isbn,
  item.quantity,
  CAST(item.unit_price AS DECIMAL(10, 2)) AS unit_price
FROM bronze.instore_orders
  LATERAL VIEW EXPLODE(
    FROM_JSON(items, 'ARRAY<STRUCT<isbn: STRING, title: STRING, quantity: INT, unit_price: DOUBLE>>')
  ) t AS item

Merge the exploded order items into `silver.order_items` using the `order_items_exploded` temporary view created above, matching on the composite key `(order_id, order_channel, isbn)`.

In [0]:
-- @test:silver_order_items_merge
-- TODO: Write SQL to implement the silver_order_items_merge transformation
merge into silver.order_items t
using order_items_exploded s
on t.order_id = s.order_id and t.order_channel = s.order_channel and t.isbn = s.isbn
when not matched then insert *
-- @test:silver

---
## Step 7: Verify

Now let's run some checks to make sure the Silver layer is correct.

**Row counts** — Compare Silver table counts to their Bronze sources. `silver.books` may be slightly less than `bronze.books` if any rows were rejected by the quality checks.

In [0]:
SELECT 'bronze.categories' AS table_name, COUNT(*) AS row_count FROM bronze.categories
UNION ALL
SELECT 'silver.categories', COUNT(*) FROM silver.categories
UNION ALL
SELECT 'bronze.stores', COUNT(*) FROM bronze.stores
UNION ALL
SELECT 'silver.stores', COUNT(*) FROM silver.stores
UNION ALL
SELECT 'bronze.books', COUNT(*) FROM bronze.books
UNION ALL
SELECT 'silver.books', COUNT(*) FROM silver.books
UNION ALL
SELECT 'silver.customers', COUNT(*) FROM silver.customers
UNION ALL
SELECT 'silver.orders', COUNT(*) FROM silver.orders
UNION ALL
SELECT 'silver.order_items', COUNT(*) FROM silver.order_items

**Customer count** — The number of customers should equal the number of distinct `customer_email` values in `bronze.online_orders`.

In [0]:
SELECT
  (SELECT COUNT(*) FROM silver.customers) AS silver_customer_count,
  (SELECT COUNT(DISTINCT customer_email) FROM bronze.online_orders) AS bronze_distinct_emails

**Order count** — `silver.orders` should equal the combined count of both Bronze order tables.

In [0]:
SELECT
  (SELECT COUNT(*) FROM silver.orders) AS silver_order_count,
  (SELECT COUNT(*) FROM bronze.online_orders) + (SELECT COUNT(*) FROM bronze.instore_orders) AS bronze_combined_count

**Referential integrity** — Every `customer_email` in `silver.orders` should exist in `silver.customers` (or be the sentinel value `'in-store'`). This query should return **no rows**.

In [0]:
SELECT DISTINCT o.customer_email
FROM silver.orders o
LEFT JOIN silver.customers c ON o.customer_email = c.email
WHERE c.email IS NULL
  AND o.customer_email != 'in-store'

**Referential integrity** — Every `isbn` in `silver.order_items` should exist in `silver.books`. This query should return **no rows**.

In [0]:
SELECT DISTINCT oi.isbn
FROM silver.order_items oi
LEFT JOIN silver.books b ON oi.isbn = b.isbn
WHERE b.isbn IS NULL

**Referential integrity** — Every `category_id` in `silver.books` should exist in `silver.categories`. This query should return **no rows**.

In [0]:
SELECT DISTINCT b.category_id
FROM silver.books b
LEFT JOIN silver.categories c ON b.category_id = c.category_id
WHERE c.category_id IS NULL

**Referential integrity** — Every `parent_category_id` in `silver.categories` should either be empty (top-level) or reference another category. This query should return **no rows**.

In [0]:
SELECT child.category_id, child.category_name, child.parent_category_id
FROM silver.categories child
LEFT JOIN silver.categories parent ON child.parent_category_id = parent.category_id
WHERE child.parent_category_id != ''
  AND parent.category_id IS NULL

**Total amount cross-check** — For each order, verify that the sum of `quantity * unit_price` across its line items matches the `total_amount` stored on the order. Any mismatches indicate a data quality issue.

In [0]:
SELECT
  o.order_id,
  o.order_channel,
  o.total_amount AS order_total,
  SUM(oi.quantity * oi.unit_price) AS computed_total,
  o.total_amount - SUM(oi.quantity * oi.unit_price) AS difference
FROM silver.orders o
JOIN silver.order_items oi
  ON o.order_id = oi.order_id
  AND o.order_channel = oi.order_channel
GROUP BY o.order_id, o.order_channel, o.total_amount
HAVING o.total_amount != SUM(oi.quantity * oi.unit_price)